##### Setup
- Same column-by-column checks as `column_check.ipynb`, but run against the cleaned data (`df_clean_part*.parquet` from `clean.ipynb`) instead of the raw CSVs

In [2]:
import duckdb
import pandas as pd

CLEAN = "read_parquet('df_clean_part*.parquet')"

##### Check: SVC_USE_DAYS_GRP (Ordinal)

In [2]:
svc_use_days_grp_values = duckdb.sql(f"""
    SELECT SVC_USE_DAYS_GRP, count(*) AS n
    FROM {CLEAN}
    GROUP BY SVC_USE_DAYS_GRP
    ORDER BY n DESC
""").df()

svc_use_days_grp_values

,SVC_USE_DAYS_GRP,n
0,36개월 이상,15160536
1,12개월~24개월미만,2271681
2,24개월~ 36개월미만,2233593
3,6개월미만,1232522
4,6개월~12개월미만,1155232


##### Check: MEDIA_NM_GRP (Nominal)

In [3]:
media_nm_grp_values = duckdb.sql(f"""
    SELECT MEDIA_NM_GRP, count(*) AS n
    FROM {CLEAN}
    GROUP BY MEDIA_NM_GRP
    ORDER BY n DESC
""").df()

media_nm_grp_values

,MEDIA_NM_GRP,n
0,HD,16141579
1,UHD,5840870
2,기타,71115


##### Check: PROD_NM_GRP (Ordinal)

In [4]:
prod_nm_grp_values = duckdb.sql(f"""
    SELECT PROD_NM_GRP, count(*) AS n
    FROM {CLEAN}
    GROUP BY PROD_NM_GRP
    ORDER BY n DESC
""").df()

prod_nm_grp_values

,PROD_NM_GRP,n
0,베이직,11336429
1,이코노미,6218707
2,프리미엄,4311490
3,스탠다드,185211
4,세이버,1652
5,기타,75


##### Check: PROD_OLD_YN (Categorical, Binary)

In [5]:
prod_old_yn_values = duckdb.sql(f"""
    SELECT PROD_OLD_YN, count(*) AS n
    FROM {CLEAN}
    GROUP BY PROD_OLD_YN
    ORDER BY n DESC
""").df()

prod_old_yn_values

,PROD_OLD_YN,n
0,N,21873300
1,Y,180264


##### Check: PROD_ONE_PLUS_YN (Categorical, Binary)

In [6]:
prod_one_plus_yn_values = duckdb.sql(f"""
    SELECT PROD_ONE_PLUS_YN, count(*) AS n
    FROM {CLEAN}
    GROUP BY PROD_ONE_PLUS_YN
    ORDER BY n DESC
""").df()

prod_one_plus_yn_values

,PROD_ONE_PLUS_YN,n
0,N,17326121
1,Y,4727443


##### Check: AGMT_KIND_NM (Categorical, Nominal; '정보없음' should now be NULL)

In [7]:
agmt_kind_nm_values = duckdb.sql(f"""
    SELECT AGMT_KIND_NM, count(*) AS n
    FROM {CLEAN}
    GROUP BY AGMT_KIND_NM
    ORDER BY n DESC
""").df()

agmt_kind_nm_values

,AGMT_KIND_NM,n
0,재약정,9423314
1,신규,7925123
2,약정승계,4633840
3,약정연장,42549
4,약정축소,26588
5,약정갱신,2073
6,None,77


##### Check: STB_RES_1M_YN (Categorical, Binary; '""' rows should be gone)

In [8]:
stb_res_1m_yn_values = duckdb.sql(f"""
    SELECT STB_RES_1M_YN, count(*) AS n
    FROM {CLEAN}
    GROUP BY STB_RES_1M_YN
    ORDER BY n DESC
""").df()

stb_res_1m_yn_values

,STB_RES_1M_YN,n
0,N,19515363
1,Y,2538201


##### Check: SVOD_SCRB_CNT_GRP (Ordinal)

In [9]:
svod_scrb_cnt_grp_values = duckdb.sql(f"""
    SELECT SVOD_SCRB_CNT_GRP, count(*) AS n
    FROM {CLEAN}
    GROUP BY SVOD_SCRB_CNT_GRP
    ORDER BY n DESC
""").df()

svod_scrb_cnt_grp_values

,SVOD_SCRB_CNT_GRP,n
0,0건,21537026
1,1건,405126
2,2건,69321
3,3건 이상,42091


##### Check: PAID_CHNL_CNT_GRP (Ordinal)

In [10]:
paid_chnl_cnt_grp_values = duckdb.sql(f"""
    SELECT PAID_CHNL_CNT_GRP, count(*) AS n
    FROM {CLEAN}
    GROUP BY PAID_CHNL_CNT_GRP
    ORDER BY n DESC
""").df()

paid_chnl_cnt_grp_values

,PAID_CHNL_CNT_GRP,n
0,0건,21240803
1,1건,717886
2,2건,72728
3,3건 이상,22147


##### Check: SCRB_PATH_NM_GRP (Categorical, Nominal; '정보없음' should now be NULL)
- **→ `clean.ipynb` update**: found here via `broad_sentinel_check`, then fixed in `clean.ipynb` (cells `a4f36c37`/`530e6a78`), which patches the saved Parquet parts in place, converting `'정보없음'` → NULL. Already run — result below reflects the fix.

In [11]:
scrb_path_nm_grp_values = duckdb.sql(f"""
    SELECT SCRB_PATH_NM_GRP, count(*) AS n
    FROM {CLEAN}
    GROUP BY SCRB_PATH_NM_GRP
    ORDER BY n DESC
""").df()

scrb_path_nm_grp_values

,SCRB_PATH_NM_GRP,n
0,현장경로,8225308
1,O/B,5453621
2,I/B,4288227
3,일반상담,3094858
4,직영몰,778180
5,임직원,102930
6,None,39254
7,기타,39192
8,전략채널,31208
9,렌탈제휴,786


##### Check: INHOME_RATE (Numeric; '알수없음' / '""' still unresolved as of this writing)
- **→ `clean.ipynb` update**: fix added in `clean.ipynb` (cells `b70abf92`/`ec5c0ef5`) — converts `'알수없음'` → NULL and casts the column to `DOUBLE`. Also checked here whether `'알수없음'` correlates with new customers (short `TOTAL_USED_DAYS`) — it does **not** (difference was small and in the opposite direction), so this stays a plain NULL, no separate category needed. **Not yet run as of this writing** — the result below still shows the raw `'알수없음'` string; re-run `ec5c0ef5` in `clean.ipynb` then re-run this cell to see the fix applied.

In [12]:
inhome_rate_values = duckdb.sql(f"""
    SELECT INHOME_RATE, count(*) AS n
    FROM {CLEAN}
    GROUP BY INHOME_RATE
    ORDER BY n DESC
""").df()

inhome_rate_values

,INHOME_RATE,n
0,10.0,3905474
1,0.0,3540010
2,20.0,3469282
3,알수없음,3030808
4,30.0,2891071
5,40.0,2232435
6,50.0,1561090
7,60.0,904906
8,70.0,380620
9,80.0,117562


##### Why Does '알수없음' Happen? Check Against TOTAL_USED_DAYS
- `INHOME_RATE` is a calculated metric, so '알수없음' (unknown) may occur for new customers who haven't accumulated enough usage history yet
- If `TOTAL_USED_DAYS` (days since subscription) is meaningfully shorter for the '알수없음' group, that supports "calculation wasn't possible yet" rather than "a distinct segment" — backing up the decision to convert it to NULL

In [4]:
inhome_rate_unknown_check = duckdb.sql(f"""
    SELECT
        CASE WHEN INHOME_RATE = '알수없음' THEN '알수없음' ELSE '값있음' END AS group_label,
        count(*) AS n_rows,
        round(avg(TRY_CAST(TOTAL_USED_DAYS AS DOUBLE)), 1) AS avg_total_used_days,
        approx_quantile(TRY_CAST(TOTAL_USED_DAYS AS DOUBLE), 0.5) AS median_total_used_days,
        count(*) FILTER (WHERE TRY_CAST(TOTAL_USED_DAYS AS DOUBLE) <= 90) AS n_new_customers,
        round(count(*) FILTER (WHERE TRY_CAST(TOTAL_USED_DAYS AS DOUBLE) <= 90) * 100.0 / count(*), 2) AS pct_new_customers
    FROM {CLEAN}
    GROUP BY group_label
""").df()

inhome_rate_unknown_check

,group_label,n_rows,avg_total_used_days,median_total_used_days,n_new_customers,pct_new_customers
0,값있음,19022756,2747.0,2906.983457,224035,1.18
1,알수없음,3030808,2831.4,2976.513357,16632,0.55


##### Check: AGMT_END_SEG (Ordinal)

In [13]:
agmt_end_seg_values = duckdb.sql(f"""
    SELECT AGMT_END_SEG, count(*) AS n
    FROM {CLEAN}
    GROUP BY AGMT_END_SEG
    ORDER BY n DESC
""").df()

agmt_end_seg_values

,AGMT_END_SEG,n
0,약정만료전 12개월이상,9388587
1,약정만료후 12개월이상,7772274
2,약정만료전 9~12개월,1078669
3,약정만료전 6~9개월,958498
4,약정만료전 3~6개월,795058
5,약정만료 1개월,328269
6,약정만료후 3~6개월,318513
7,약정만료후 6~9개월,279563
8,약정만료후 9~12개월,276619
9,약정만료전 2~3개월,224681


##### Check: AGMT_END_YMD (Date; '정보없음' should now be NULL, '무약정' kept)

In [14]:
agmt_end_ymd_summary = duckdb.sql(f"""
    SELECT
        CASE
            WHEN AGMT_END_YMD IS NULL THEN 'NULL'
            WHEN AGMT_END_YMD = '무약정' THEN '무약정'
            ELSE '실제 날짜'
        END AS value_type,
        count(*) AS n,
        min(TRY_STRPTIME(AGMT_END_YMD, '%Y%m%d')) AS min_date,
        max(TRY_STRPTIME(AGMT_END_YMD, '%Y%m%d')) AS max_date
    FROM {CLEAN}
    GROUP BY value_type
    ORDER BY n DESC
""").df()

agmt_end_ymd_summary

,value_type,n,min_date,max_date
0,실제 날짜,21880439,2004-10-13,2027-12-30
1,무약정,173091,NaT,NaT
2,NULL,34,NaT,NaT


##### Check: TOTAL_USED_DAYS (Numeric, Ratio scale)

In [15]:
total_used_days_summary = duckdb.sql(f"""
    SELECT
        count(*) AS n,
        min(TRY_CAST(TOTAL_USED_DAYS AS DOUBLE)) AS min_v,
        approx_quantile(TRY_CAST(TOTAL_USED_DAYS AS DOUBLE), 0.5) AS median_v,
        approx_quantile(TRY_CAST(TOTAL_USED_DAYS AS DOUBLE), 0.99) AS p99,
        approx_quantile(TRY_CAST(TOTAL_USED_DAYS AS DOUBLE), 0.999) AS p999,
        max(TRY_CAST(TOTAL_USED_DAYS AS DOUBLE)) AS max_v,
        count(*) FILTER (WHERE TRY_CAST(TOTAL_USED_DAYS AS DOUBLE) > 15000) AS over_15000_days
    FROM {CLEAN}
""").df()

total_used_days_summary

,n,min_v,median_v,p99,p999,max_v,over_15000_days
0,22053564,0.0,2918.585999,5305.137684,9921.180984,45289.0,415


##### Check: TV_SCRB (Numeric, Ratio scale)

In [16]:
tv_scrb_summary = duckdb.sql(f"""
    SELECT
        count(*) AS n,
        min(TRY_CAST(TV_SCRB AS DOUBLE)) AS min_v,
        avg(TRY_CAST(TV_SCRB AS DOUBLE)) AS avg_v,
        approx_quantile(TRY_CAST(TV_SCRB AS DOUBLE), 0.5) AS median_v,
        max(TRY_CAST(TV_SCRB AS DOUBLE)) AS max_v
    FROM {CLEAN}
""").df()

tv_scrb_summary

,n,min_v,avg_v,median_v,max_v
0,22053564,1.0,1.779614,1.793033,235.0


##### Check: ANALOG_SCRB (Numeric, Ratio scale)

In [17]:
analog_scrb_summary = duckdb.sql(f"""
    SELECT
        count(*) AS n,
        min(TRY_CAST(ANALOG_SCRB AS DOUBLE)) AS min_v,
        avg(TRY_CAST(ANALOG_SCRB AS DOUBLE)) AS avg_v,
        approx_quantile(TRY_CAST(ANALOG_SCRB AS DOUBLE), 0.5) AS median_v,
        max(TRY_CAST(ANALOG_SCRB AS DOUBLE)) AS max_v
    FROM {CLEAN}
""").df()

analog_scrb_summary

,n,min_v,avg_v,median_v,max_v
0,22053564,0.0,0.020688,0.0,29.0


##### Check: DIGITAL_SCRB (Numeric, Ratio scale)

In [18]:
digital_scrb_summary = duckdb.sql(f"""
    SELECT
        count(*) AS n,
        min(TRY_CAST(DIGITAL_SCRB AS DOUBLE)) AS min_v,
        avg(TRY_CAST(DIGITAL_SCRB AS DOUBLE)) AS avg_v,
        approx_quantile(TRY_CAST(DIGITAL_SCRB AS DOUBLE), 0.5) AS median_v,
        max(TRY_CAST(DIGITAL_SCRB AS DOUBLE)) AS max_v
    FROM {CLEAN}
""").df()

digital_scrb_summary

,n,min_v,avg_v,median_v,max_v
0,22053564,1.0,1.758926,1.302754,235.0


##### Check: TOTAL_INTERNET_SCRB (Numeric, Ratio scale)

In [19]:
total_internet_scrb_summary = duckdb.sql(f"""
    SELECT
        count(*) AS n,
        min(TRY_CAST(TOTAL_INTERNET_SCRB AS DOUBLE)) AS min_v,
        avg(TRY_CAST(TOTAL_INTERNET_SCRB AS DOUBLE)) AS avg_v,
        approx_quantile(TRY_CAST(TOTAL_INTERNET_SCRB AS DOUBLE), 0.5) AS median_v,
        max(TRY_CAST(TOTAL_INTERNET_SCRB AS DOUBLE)) AS max_v
    FROM {CLEAN}
""").df()

total_internet_scrb_summary

,n,min_v,avg_v,median_v,max_v
0,22053564,0.0,0.495286,0.0,102.0


##### Check: GIGA_INTERNET_SCRB (Numeric, Ratio scale)

In [20]:
giga_internet_scrb_summary = duckdb.sql(f"""
    SELECT
        count(*) AS n,
        min(TRY_CAST(GIGA_INTERNET_SCRB AS DOUBLE)) AS min_v,
        avg(TRY_CAST(GIGA_INTERNET_SCRB AS DOUBLE)) AS avg_v,
        approx_quantile(TRY_CAST(GIGA_INTERNET_SCRB AS DOUBLE), 0.5) AS median_v,
        max(TRY_CAST(GIGA_INTERNET_SCRB AS DOUBLE)) AS max_v
    FROM {CLEAN}
""").df()

giga_internet_scrb_summary

,n,min_v,avg_v,median_v,max_v
0,22053564,0.0,0.134082,0.0,21.0


##### Check: BUNDLE_YN (Categorical, Binary)

In [21]:
bundle_yn_values = duckdb.sql(f"""
    SELECT BUNDLE_YN, count(*) AS n
    FROM {CLEAN}
    GROUP BY BUNDLE_YN
    ORDER BY n DESC
""").df()

bundle_yn_values

,BUNDLE_YN,n
0,N,12330870
1,Y,9722694


##### Check: DIGITAL_GIGA_YN (Categorical, Binary)

In [22]:
digital_giga_yn_values = duckdb.sql(f"""
    SELECT DIGITAL_GIGA_YN, count(*) AS n
    FROM {CLEAN}
    GROUP BY DIGITAL_GIGA_YN
    ORDER BY n DESC
""").df()

digital_giga_yn_values

,DIGITAL_GIGA_YN,n
0,N,19180501
1,Y,2873063


##### Check: DIGITAL_ALOG_YN (Categorical, Binary)

In [23]:
digital_alog_yn_values = duckdb.sql(f"""
    SELECT DIGITAL_ALOG_YN, count(*) AS n
    FROM {CLEAN}
    GROUP BY DIGITAL_ALOG_YN
    ORDER BY n DESC
""").df()

digital_alog_yn_values

,DIGITAL_ALOG_YN,n
0,N,21649259
1,Y,404305


##### Check: TV_I_CNT (Numeric, Ratio scale)

In [24]:
tv_i_cnt_summary = duckdb.sql(f"""
    SELECT
        count(*) AS n,
        min(TRY_CAST(TV_I_CNT AS DOUBLE)) AS min_v,
        avg(TRY_CAST(TV_I_CNT AS DOUBLE)) AS avg_v,
        approx_quantile(TRY_CAST(TV_I_CNT AS DOUBLE), 0.5) AS median_v,
        max(TRY_CAST(TV_I_CNT AS DOUBLE)) AS max_v
    FROM {CLEAN}
""").df()

tv_i_cnt_summary

,n,min_v,avg_v,median_v,max_v
0,22053564,1.0,2.2749,2.0,275.0


##### Check: CH_LAST_DAYS_BF_GRP (Ordinal)

In [25]:
ch_last_days_bf_grp_values = duckdb.sql(f"""
    SELECT CH_LAST_DAYS_BF_GRP, count(*) AS n
    FROM {CLEAN}
    GROUP BY CH_LAST_DAYS_BF_GRP
    ORDER BY n DESC
""").df()

ch_last_days_bf_grp_values

,CH_LAST_DAYS_BF_GRP,n
0,일주일내,17188739
1,3개월내없음,3629629
2,일주일전,592691
3,2주일전,324687
4,3주일전,232920
5,4주일전,84898


##### Check: VOC_TOTAL_MONTH1_YN (Categorical, Binary)

In [26]:
voc_total_month1_yn_values = duckdb.sql(f"""
    SELECT VOC_TOTAL_MONTH1_YN, count(*) AS n
    FROM {CLEAN}
    GROUP BY VOC_TOTAL_MONTH1_YN
    ORDER BY n DESC
""").df()

voc_total_month1_yn_values

,VOC_TOTAL_MONTH1_YN,n
0,N,18770724
1,Y,3282840


##### Check: VOC_STOP_CANCEL_MONTH1_YN (Categorical, Binary)

In [27]:
voc_stop_cancel_month1_yn_values = duckdb.sql(f"""
    SELECT VOC_STOP_CANCEL_MONTH1_YN, count(*) AS n
    FROM {CLEAN}
    GROUP BY VOC_STOP_CANCEL_MONTH1_YN
    ORDER BY n DESC
""").df()

voc_stop_cancel_month1_yn_values

,VOC_STOP_CANCEL_MONTH1_YN,n
0,N,21588646
1,Y,464918


##### Check: AGE_GRP10 (Ordinal; '""'/'10대미만' rows should be gone, '연령없음' kept as its own category)
- **→ `clean.ipynb` update**: none — decision was to **keep** `'연령없음'` as its own category (not convert to NULL)
- **Row-level test**: 8.32x higher rate of `TV_I_CNT >= 10` for `연령없음` vs normal (z=158.28, p≈0, 95% CI [5.37%, 5.76%])
- **Customer-level re-check** (fixes the row/customer independence violation — same customer appears in up to 11 monthly rows): 6,203 `연령없음` customers vs 2,129,058 normal customers, rate 6.46% vs 0.99% → **6.51x**, z=43.04, p≈0, 95% CI [4.86%, 6.08%]. Effect size shrank somewhat (8.32x → 6.51x, as expected once the inflated row-level SE is corrected) but the conclusion holds — this is a real, statistically robust segment (likely business/bulk accounts), not an artifact of counting the same customers repeatedly. Average repeat count is nearly identical between groups (연령없음: 10.06 months, normal: 10.33 months), so the row-level test wasn't disproportionately inflated by one group

In [28]:
age_grp10_values = duckdb.sql(f"""
    SELECT AGE_GRP10, count(*) AS n
    FROM {CLEAN}
    GROUP BY AGE_GRP10
    ORDER BY n DESC
""").df()

age_grp10_values

,AGE_GRP10,n
0,60대,6227578
1,50대,4586109
2,70대,4467679
3,40대,2772524
4,80대,2455740
5,30대,963215
6,90대이상,332348
7,20대,185727
8,연령없음,62430
9,10대,214


##### Is AGE_GRP10 = '연령없음' a Business/Bulk Account Segment?
- Same rate-based test used for the `sha2_hash+p_mt` conflict investigation: compare the % of rows with `TV_I_CNT >= 10` between the `연령없음` group and everyone else
- If `연령없음` behaves like a corporate/institutional account (no personal age applies), it should show a meaningfully higher rate of large-TV_I_CNT accounts

In [40]:
age_grp10_bulk_check = duckdb.sql(f"""
    WITH labeled AS (
        SELECT
            TRY_CAST(TV_I_CNT AS DOUBLE) AS tv_i_cnt,
            CASE WHEN AGE_GRP10 = '연령없음' THEN '연령없음' ELSE 'normal' END AS group_label
        FROM {CLEAN}
    )
    SELECT
        group_label,
        count(*) AS n_rows,
        round(avg(tv_i_cnt), 2) AS avg_tv_i_cnt,
        max(tv_i_cnt) AS max_tv_i_cnt,
        count(*) FILTER (WHERE tv_i_cnt >= 10) AS n_rows_bulk,
        round(count(*) FILTER (WHERE tv_i_cnt >= 10) * 100.0 / count(*), 4) AS pct_rows_bulk
    FROM labeled
    GROUP BY group_label
""").df()

age_grp10_bulk_check

,group_label,n_rows,avg_tv_i_cnt,max_tv_i_cnt,n_rows_bulk,pct_rows_bulk
0,연령없음,62430,4.83,91.0,3948,6.3239
1,normal,21991134,2.27,275.0,167051,0.7596


##### Two-Proportion Z-Test: Is the Gap Statistically Significant?
- Tests whether the difference in `pct_rows_bulk` between `연령없음` and `normal` could plausibly be due to chance, accounting for the very different sample sizes
- `z > ~1.96` conventionally means "statistically significant at p < 0.05" — bigger `|z|` means stronger evidence the two rates are genuinely different

In [41]:
import math

row_a = age_grp10_bulk_check.loc[age_grp10_bulk_check.group_label == '연령없음'].iloc[0]
row_b = age_grp10_bulk_check.loc[age_grp10_bulk_check.group_label == 'normal'].iloc[0]

n1, x1 = row_a["n_rows"], row_a["n_rows_bulk"]
n2, x2 = row_b["n_rows"], row_b["n_rows_bulk"]

p1 = x1 / n1
p2 = x2 / n2
pooled_p = (x1 + x2) / (n1 + n2)
se = math.sqrt(pooled_p * (1 - pooled_p) * (1 / n1 + 1 / n2))
z = (p1 - p2) / se
p_value = math.erfc(abs(z) / math.sqrt(2))  # two-sided p-value, no scipy needed

print(f"연령없음 rate: {p1:.4%} (n={n1:,})")
print(f"normal rate:   {p2:.4%} (n={n2:,})")
print(f"z-score: {z:.2f}")
print(f"p-value: {p_value:.3e}")

연령없음 rate: 6.3239% (n=62,430)
normal rate:   0.7596% (n=21,991,134)
z-score: 158.28
p-value: 0.000e+00


##### 95% Confidence Interval for the Rate Difference
- A p-value alone can't distinguish "huge sample made a trivial difference look significant" from "the difference is actually large" — a confidence interval shows the *magnitude* directly
- Uses the unpooled standard error (standard for a CI, vs. the pooled SE used for the hypothesis test above)

In [ ]:
z_critical = 1.96  # 95% CI

se_unpooled = math.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
diff = p1 - p2
ci_low = diff - z_critical * se_unpooled
ci_high = diff + z_critical * se_unpooled

print(f"비율 차이 (연령없음 - normal): {diff:.4%}")
print(f"95% 신뢰구간: [{ci_low:.4%}, {ci_high:.4%}]")
print(f"상대 배율 (연령없음 / normal): {p1 / p2:.2f}x")

비율 차이 (연령없음 - normal): 5.5643%
95% 신뢰구간: [5.3733%, 5.7552%]
상대 배율 (연령없음 / normal): 8.32x


##### Re-Check at Customer Level (Independence Fix)
- The row-level test above treats each of the ~22M rows as independent, but the same customer appears in up to 11 monthly rows — violating the independence assumption behind the z-test/CI
- Collapse to one row per `sha2_hash` (their most recent snapshot) and redo the same rate comparison, z-test, and CI — if the conclusion holds up here too, the row-level result wasn't just an artifact of double-counting the same customers

In [5]:
customer_level_check = duckdb.sql(f"""
    WITH per_customer AS (
        SELECT
            sha2_hash,
            arg_max(AGE_GRP10, p_mt) AS AGE_GRP10,
            arg_max(TRY_CAST(TV_I_CNT AS DOUBLE), p_mt) AS tv_i_cnt
        FROM {CLEAN}
        GROUP BY sha2_hash
    ),
    labeled AS (
        SELECT
            tv_i_cnt,
            CASE WHEN AGE_GRP10 = '연령없음' THEN '연령없음' ELSE 'normal' END AS group_label
        FROM per_customer
    )
    SELECT
        group_label,
        count(*) AS n_customers,
        round(avg(tv_i_cnt), 2) AS avg_tv_i_cnt,
        count(*) FILTER (WHERE tv_i_cnt >= 10) AS n_bulk,
        round(count(*) FILTER (WHERE tv_i_cnt >= 10) * 100.0 / count(*), 4) AS pct_bulk
    FROM labeled
    GROUP BY group_label
""").df()

customer_level_check

,group_label,n_customers,avg_tv_i_cnt,n_bulk,pct_bulk
0,연령없음,6203,4.68,401,6.4646
1,normal,2129058,2.36,21157,0.9937


In [6]:
import math

row_a2 = customer_level_check.loc[customer_level_check.group_label == '연령없음'].iloc[0]
row_b2 = customer_level_check.loc[customer_level_check.group_label == 'normal'].iloc[0]

n1c, x1c = row_a2["n_customers"], row_a2["n_bulk"]
n2c, x2c = row_b2["n_customers"], row_b2["n_bulk"]

p1c = x1c / n1c
p2c = x2c / n2c
pooled_pc = (x1c + x2c) / (n1c + n2c)
se_c = math.sqrt(pooled_pc * (1 - pooled_pc) * (1 / n1c + 1 / n2c))
z_c = (p1c - p2c) / se_c
p_value_c = math.erfc(abs(z_c) / math.sqrt(2))

se_unpooled_c = math.sqrt(p1c * (1 - p1c) / n1c + p2c * (1 - p2c) / n2c)
diff_c = p1c - p2c
ci_low_c = diff_c - 1.96 * se_unpooled_c
ci_high_c = diff_c + 1.96 * se_unpooled_c

print(f"[고객 단위] 연령없음 rate: {p1c:.4%} (n={n1c:,} customers, was 62,430 rows)")
print(f"[고객 단위] normal rate:   {p2c:.4%} (n={n2c:,} customers, was 21,991,134 rows)")
print(f"[고객 단위] z-score: {z_c:.2f}, p-value: {p_value_c:.3e}")
print(f"[고객 단위] 95% CI: [{ci_low_c:.4%}, {ci_high_c:.4%}]")
print(f"[고객 단위] 상대 배율: {p1c / p2c:.2f}x")
print()
print(f"평균 반복 횟수 (연령없음): {62430 / n1c:.2f}개월")
print(f"평균 반복 횟수 (normal):   {21991134 / n2c:.2f}개월")

[고객 단위] 연령없음 rate: 6.4646% (n=6,203 customers, was 62,430 rows)
[고객 단위] normal rate:   0.9937% (n=2,129,058 customers, was 21,991,134 rows)
[고객 단위] z-score: 43.04, p-value: 0.000e+00
[고객 단위] 95% CI: [4.8588%, 6.0830%]
[고객 단위] 상대 배율: 6.51x

평균 반복 횟수 (연령없음): 10.06개월
평균 반복 횟수 (normal):   10.33개월


##### Check: EMAIL_RECV_CLS_NM (Nominal)

In [29]:
email_recv_cls_nm_values = duckdb.sql(f"""
    SELECT EMAIL_RECV_CLS_NM, count(*) AS n
    FROM {CLEAN}
    GROUP BY EMAIL_RECV_CLS_NM
    ORDER BY n DESC
""").df()

email_recv_cls_nm_values

,EMAIL_RECV_CLS_NM,n
0,수신,13873763
1,전체거부,7677726
2,미응답,449814
3,광고거부,52261


##### Check: SMS_SEND_CLS_NM (Nominal)

In [30]:
sms_send_cls_nm_values = duckdb.sql(f"""
    SELECT SMS_SEND_CLS_NM, count(*) AS n
    FROM {CLEAN}
    GROUP BY SMS_SEND_CLS_NM
    ORDER BY n DESC
""").df()

sms_send_cls_nm_values

,SMS_SEND_CLS_NM,n
0,수신,16701148
1,전체거부,4791561
2,광고거부,341932
3,미응답,218923


##### Check: CH_HH_AVG_MONTH1 (Numeric, Ratio scale)

In [31]:
ch_hh_avg_month1_summary = duckdb.sql(f"""
    SELECT
        count(*) AS n,
        min(TRY_CAST(CH_HH_AVG_MONTH1 AS DOUBLE)) AS min_v,
        avg(TRY_CAST(CH_HH_AVG_MONTH1 AS DOUBLE)) AS avg_v,
        approx_quantile(TRY_CAST(CH_HH_AVG_MONTH1 AS DOUBLE), 0.5) AS median_v,
        max(TRY_CAST(CH_HH_AVG_MONTH1 AS DOUBLE)) AS max_v
    FROM {CLEAN}
""").df()

ch_hh_avg_month1_summary

,n,min_v,avg_v,median_v,max_v
0,22053564,0.0,4.556711,3.627387,35.39


##### Check: CH_25_RATIO_MONTH1 (Numeric, Ratio scale; capped at 100 during cleaning)

In [32]:
ch_25_ratio_month1_summary = duckdb.sql(f"""
    SELECT
        count(*) AS n,
        min(TRY_CAST(CH_25_RATIO_MONTH1 AS DOUBLE)) AS min_v,
        avg(TRY_CAST(CH_25_RATIO_MONTH1 AS DOUBLE)) AS avg_v,
        approx_quantile(TRY_CAST(CH_25_RATIO_MONTH1 AS DOUBLE), 0.5) AS median_v,
        max(TRY_CAST(CH_25_RATIO_MONTH1 AS DOUBLE)) AS max_v
    FROM {CLEAN}
""").df()

ch_25_ratio_month1_summary

,n,min_v,avg_v,median_v,max_v
0,22053564,0.0,2.410497,0.830725,100.0


##### Check: CH_25_RATIO_MEAN_3MM (Numeric, Ratio scale)

In [33]:
ch_25_ratio_mean_3mm_summary = duckdb.sql(f"""
    SELECT
        count(*) AS n,
        min(TRY_CAST(CH_25_RATIO_MEAN_3MM AS DOUBLE)) AS min_v,
        avg(TRY_CAST(CH_25_RATIO_MEAN_3MM AS DOUBLE)) AS avg_v,
        approx_quantile(TRY_CAST(CH_25_RATIO_MEAN_3MM AS DOUBLE), 0.5) AS median_v,
        max(TRY_CAST(CH_25_RATIO_MEAN_3MM AS DOUBLE)) AS max_v
    FROM {CLEAN}
""").df()

ch_25_ratio_mean_3mm_summary

,n,min_v,avg_v,median_v,max_v
0,22053564,0.0,2.410243,0.83093,100.0


##### Check: CH_FAV_RNK1 (Categorical, Nominal)

In [34]:
ch_fav_rnk1_values = duckdb.sql(f"""
    SELECT CH_FAV_RNK1, count(*) AS n
    FROM {CLEAN}
    GROUP BY CH_FAV_RNK1
    ORDER BY n DESC
""").df()

ch_fav_rnk1_values

,CH_FAV_RNK1,n
0,기타,13541647
1,KBS1,2557348
2,KBS2,1074858
3,TV조선,753412
4,SBS,703995
5,연합뉴스TV,651362
6,MBC,639160
7,YTN,551460
8,MBN,527412
9,tvN,353554


##### Check: KIDS_USE_PV_MONTH1 (Numeric, Ratio scale; heavily right-skewed — resolved, no treatment needed)
- **→ `clean.ipynb` update**: skew diagnostics (p90/p99/p999, `pct_zero`) added and run in `clean.ipynb` (cells `bd77d2cd`/`e5d7fa6c`) — 86.03% zero, p90=1.0, p99=5.73, p999=40.63, max=4621.0. "Mostly unused, a few heavy users" pattern, not a physically-impossible value. **Decision: no treatment needed** — tree models don't assume normality, so the raw skew isn't a problem the way it would be for a linear model

In [35]:
kids_use_pv_month1_summary = duckdb.sql(f"""
    SELECT
        count(*) AS n,
        min(TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE)) AS min_v,
        avg(TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE)) AS avg_v,
        approx_quantile(TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE), 0.5) AS median_v,
        max(TRY_CAST(KIDS_USE_PV_MONTH1 AS DOUBLE)) AS max_v
    FROM {CLEAN}
""").df()

kids_use_pv_month1_summary

,n,min_v,avg_v,median_v,max_v
0,22053564,0.0,0.396931,0.0,4621.0


##### Check: NFX_USE_YN (Categorical, Binary)

In [36]:
nfx_use_yn_values = duckdb.sql(f"""
    SELECT NFX_USE_YN, count(*) AS n
    FROM {CLEAN}
    GROUP BY NFX_USE_YN
    ORDER BY n DESC
""").df()

nfx_use_yn_values

,NFX_USE_YN,n
0,N,20597394
1,Y,1456170


##### Check: YTB_USE_YN (Categorical, Binary)

In [37]:
ytb_use_yn_values = duckdb.sql(f"""
    SELECT YTB_USE_YN, count(*) AS n
    FROM {CLEAN}
    GROUP BY YTB_USE_YN
    ORDER BY n DESC
""").df()

ytb_use_yn_values

,YTB_USE_YN,n
0,N,19984459
1,Y,2069105


##### Check: p_mt (Ordinal (Period))

In [38]:
p_mt_values = duckdb.sql(f"""
    SELECT p_mt, count(*) AS n
    FROM {CLEAN}
    GROUP BY p_mt
    ORDER BY p_mt
""").df()

p_mt_values

,p_mt,n
0,202302,2012162
1,202303,2010412
2,202304,2008844
3,202305,2008663
4,202306,2007248
5,202307,2005646
6,202308,2003123
7,202309,2002536
8,202310,2001422
9,202311,1998547


##### Check: cancel_yn (Categorical, Binary (TARGET variable))

In [39]:
cancel_yn_values = duckdb.sql(f"""
    SELECT cancel_yn, count(*) AS n
    FROM {CLEAN}
    GROUP BY cancel_yn
    ORDER BY n DESC
""").df()

cancel_yn_values

,cancel_yn,n
0,유지,21163859
1,해지,889705
